# 06 · Profundidad en los 7 puntos y revisión a ciegas (S8)

**Historias:** SP-02.7 (spike de profundidad) → SP-02.9 (segunda prueba). **Kernel:** `colibri`
(el backend `vda` solo **lee** los mapas que escribió el notebook 05 con el otro kernel).

Para cada punto se lee la profundidad del ave en el mapa de su frame y se normaliza dentro de ese mapa (D-14):

`z_norm = (z_ave − z_flor) / (z_fondo − z_flor)`: 0 = flor, 1 = fondo, negativo = delante, positivo = detrás.

| Backend | Modelo | Qué hace | Dominio |
|---|---|---|---|
| `yolo26` | `yolo26{n,s,m,l,x}-depth` | frame completo, sin memoria | profundidad |
| `yolo26_crop` | `yolo26s-depth` | recorte fijo con flor, recorrido y franja de fondo | profundidad |
| `vda` | `vda-small` (Apache-2.0) | video completo, con consistencia temporal (notebook 05) | disparidad |

| Sección | Qué reproduce | Decisión |
|---|---|---|
| 1-2 | Lectura por punto y control de ruido | D-14 |
| 3 | Filtro de estabilidad (fondo < 5 %, autocorrelación a 0,5 s > 0,5) | D-13: YOLO26-Depth no sirve como z continua |
| 4 | Trayectoria 3D, tiras de recortes, HTML interactivo | D-16: la escala de z es diseño, no medida |
| 5 | Comparación con las salidas de S8 | — |
| 6 | Revisión a ciegas de i5bE (20 frames etiquetados por Yilber) | **D-18**: VDA 14/17, p = 0,013 → otra prueba |

El conjunto de puntos `delante` (D-15) está **congelado** en `datos_derivados/profundidad/`: se eligió por
visibilidad (no por z) con `historico/scripts/delante.py`, a partir de estadísticas de un notebook anterior.

**No hay verdad de terreno 3D.** El único juez entre backends es la revisión a ciegas (sección 6).

In [ ]:
%run 00_comun.ipynb

In [ ]:
DEPTH_BACKEND = "vda"     # "yolo26" | "yolo26_crop" | "vda"
DEPTH_MODEL = "vda-small"         # "yolo26s-depth" | "yolo26x-depth" | "vda-small"
VIDEO_LIST = ["i5bE"]             # lista de videos; vda necesita haber pasado antes por vda_inferir.py
POINT_SETS = ["b_salientes", "delante"]   # también "a_uniforme_tiempo"; "delante" solo existe en i5bE

In [ ]:
import ultralytics
import plotly.graph_objects as go
from scipy.stats import spearmanr

S8 = P["resultados"] / "S8_exploracion_3d"
TRAY = (P["resultados"] / "S5_tracking_union_test" / "trayectorias_prudente") if MODO == "artefactos" else (RUNS / "03" / "trayectorias_prudente")
P7 = (P["resultados"] / "S7_trayectorias_2d") if MODO == "artefactos" else (RUNS / "04")
ADOBE = S8 / "adobe"                       # trayectorias y 7 puntos de los clips Adobe (congelados)
CACHE_VDA = RUNS / "profundidad" / "cache_vda"   # mapas del notebook 05
TAG = f"{DEPTH_BACKEND}_{DEPTH_MODEL}"
OUT = RUNS / "06" / TAG; OUT.mkdir(parents=True, exist_ok=True)
PESOS_DEPTH = P["modelos"] / "depth"; PESOS_DEPTH.mkdir(parents=True, exist_ok=True)
os.chdir(PESOS_DEPTH)                      # Ultralytics busca (y descarga) aquí los *-depth.pt

GRID = (16, 9)     # IMGSZ, N_BG, MIN_MASK_PX y SMOOTH_S vienen de 00_comun
PATHS = {r.video_id: str(video_path(r.video_id)) for r in FUENTES.itertuples() if r.origen == "adobe_stock"}
# video: clave o id, referencia (nombre, rectángulo x0, y0, x1, y1) y región de fondo lejana y estable, sin ave
SCENES = {
    "i5bE": ("i5befvb5mxg", ("monarda", (0.54, 0.38, 0.69, 0.58)), ("bosque desenfocado arriba", (0.30, 0.03, 0.62, 0.16))),
    "DoJ1": ("doj1xa6ksqm", ("flor de aloe", (0.42, 0.10, 0.54, 0.72)), ("fondo magenta", (0.72, 0.10, 0.95, 0.50))),
    "Gu30": ("gu30nofexla", ("bráctea", (0.36, 0.20, 0.44, 0.70)), ("ladera verde", (0.68, 0.40, 0.95, 0.65))),
    "J4PK": ("j4pkgwgjaus", ("tronco", (0.58, 0.20, 0.90, 0.80)), ("detrás de la reja", (0.09, 0.05, 0.20, 0.42))),
    "A129": ("vv01_adobe1294608945", ("bráctea roja", (0.60, 0.15, 0.71, 0.68)), ("bosque desenfocado", (0.05, 0.10, 0.28, 0.60))),
    "A512": ("vv02_adobe512464022", ("flor tubular", (0.71, 0.40, 0.82, 0.50)), ("cielo", (0.04, 0.10, 0.22, 0.50))),
    "vv03": ("vv03_florisuga_file", ("flores moradas", (0.42, 0.25, 0.55, 0.58)), ("fondo negro", (0.75, 0.10, 0.96, 0.50))),
    "A160": ("adobestock_1600207728", ("hibisco", (0.60, 0.14, 0.78, 0.48)), ("bosque oscuro", (0.04, 0.05, 0.28, 0.40))),
}
print("backend", TAG, "| ultralytics", ultralytics.__version__)

def video_info(name):
    key = SCENES[name][0]
    if key in PATHS:
        return Path(PATHS[key]), ADOBE / "trayectorias" / f"trayectoria_{key}.csv", ADOBE / "puntos7"
    return (video_path(next(v for v in FUENTES.video_id if key in v)),
            next(TRAY.glob(f"trayectoria_*{key}*.csv")), P7)

def load_points(name, s):
    if s == "delante":
        return pd.read_csv(P["dd"] / "profundidad" / "puntos_i5bE_salientes_delante.csv")
    return pd.read_csv(video_info(name)[2] / f"{name}_{s}.csv")

def crop_for(t, far, W, H, margin=0.05):
    # recorte fijo a partir de la trayectoria, no de z: recorrido del ave + flor + una franja de la región de fondo
    o = t[~t.es_puente.astype(bool)]; ref = SCENES_NOW[1][1]
    x0, x1 = min((o.cx_px - o.w_px / 2).min(), ref[0] * W) - margin * W, max((o.cx_px + o.w_px / 2).max(), ref[2] * W) + margin * W
    y0, y1 = min((o.cy_px - o.h_px / 2).min(), far[1] * H) - margin * H, max((o.cy_px + o.h_px / 2).max(), ref[3] * H) + margin * H
    return max(0, int(x0)), max(0, int(y0)), min(W, int(x1)), min(H, int(y1))

## 1. Lectura por punto

Mismas reglas para todos los backends: mediana en la máscara de movimiento (recorte central si tiene menos de 30 px), flor y fondo en el propio mapa, y el mismo control de vecinos ±2. En `yolo26_crop`, la región de fondo se recorta a lo que cae dentro del recorte.

In [ ]:
R, INFO, CTX = {}, {}, {}
for name in VIDEO_LIST:
    SCENES_NOW = SCENES[name]
    path, tf, _ = video_info(name); t = pd.read_csv(tf).set_index("frame")
    cap = cv2.VideoCapture(str(path)); fps, n, W, H = cap.get(5), int(cap.get(7)), int(cap.get(3)), int(cap.get(4)); bg = background(cap, n)
    key, (rn, rr), (fn, ff) = SCENES[name]
    crop = crop_for(t.reset_index(), ff, W, H) if DEPTH_BACKEND == "yolo26_crop" else None
    be = make_backend(DEPTH_BACKEND, DEPTH_MODEL, name, (H, W), crop=crop, cache_dir=CACHE_VDA)
    ref, far = rect_px(rr, W, H), rect_px(ff, W, H)
    if crop:   # el fondo de referencia debe caer dentro del recorte
        far = (max(far[0], crop[0]), max(far[1], crop[1]), min(far[2], crop[2]), min(far[3], crop[3]))
        print(name, "recorte (px):", crop, "| fondo dentro del recorte (px):", far, f"| el recorte ocupa {100 * (crop[2] - crop[0]) / W:.0f} % del ancho")
    CTX[name] = dict(cap=cap, be=be, bg=bg, ref=ref, far=far, t=t, W=W, H=H, fps=fps, n=n, crop=crop)

    res = {}
    for s in POINT_SETS:
        PT = load_points(name, s)   # PT y no P: P son las rutas de 00_comun
        res[s] = leer_puntos_z(PT, t, cap, be, bg, ref, far, W, H)
        res[s].drop(columns=["box"]).to_csv(OUT / f"puntos_{name}_{s}.csv", index=False)
    R[name], INFO[name] = res, dict(W=W, H=H, ref=rr, ref_name=rn, path=path)
    print(name, "listo", flush=True)

## 2. Control de ruido

Regla fijada: la forma en z no es fiable si la dispersión entre vecinos es mayor que la diferencia típica entre puntos consecutivos.

In [ ]:
rows = []
for name, r in R.items():
    for s, D in r.items():
        rows.append(dict(backend=TAG, video=name, puntos=s, **control_ruido(D)))
CTRL = pd.DataFrame(rows); CTRL.to_csv(OUT / "control_ruido.csv", index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); display(CTRL.round(3))
for name, r in R.items():
    for s, D in r.items():
        print(name, s); display(D[["punto", "frame", "t_s", "x_norm", "y_norm", "z_norm", "z_norm_std", "vecinos_std", "n_px_mascara"]].round(3))

## 3. Filtro de estabilidad

Sobre todos los frames observados: variación del fondo con cámara fija (celdas sin ave) y autocorrelación a 0,5 s de la lectura suavizada. El filtro fijado pide variación < 5 % y autocorrelación > 0,5. Se mide `z_ave` (la lectura directa, comparable con la prueba anterior) y `z_norm`. En `yolo26_crop`, las celdas son solo las que caen dentro del recorte.

In [ ]:
def cells_of(d):
    H, W = d.shape; gx, gy = GRID
    with np.errstate(all="ignore"), __import__("warnings").catch_warnings():
        __import__("warnings").simplefilter("ignore")
        return np.array([[np.nanmedian(d[H * j // gy:H * (j + 1) // gy:4, W * i // gx:W * (i + 1) // gx:4]) for i in range(gx)] for j in range(gy)])

rows, SERIES = [], {}
for name in VIDEO_LIST:
    c = CTX[name]; t, be, cap, W, H = c["t"], c["be"], c["cap"], c["W"], c["H"]
    obs = t[~t.es_puente.astype(bool)]; gx, gy = GRID; A = []; S = []
    for f, r in obs.iterrows():
        fr = read_frame(cap, f); box = box_of(r, W, H); mask = motion_mask(fr, c["bg"], *box); d = be.map(f, fr)
        o = read_point(d, box, mask, c["ref"], c["far"]); cm = cells_of(d)
        free = np.ones((gy, gx), bool); free[max(0, box[1] * gy // H):min(gy, box[3] * gy // H + 1), max(0, box[0] * gx // W):min(gx, box[2] * gx // W + 1)] = False
        A.append(np.where(free, cm, np.nan)); S.append(dict(frame=f, t_s=r.t_s, segment_id=r.segment_id, z_ave=o["z_ave"], z_norm=o["z_norm"], z_flor=o["z_flor"], z_fondo=o["z_fondo"]))
    A, S = np.stack(A), pd.DataFrame(S); S.to_csv(OUT / f"serie_{name}.csv", index=False); SERIES[name] = S
    with np.errstate(all="ignore"), __import__("warnings").catch_warnings():
        __import__("warnings").simplefilter("ignore")
        cv = np.nanstd(A, axis=0) / np.nanmean(A, axis=0)
    k = max(1, int(0.5 * c["fps"]))
    def lag(col):
        w = max(5, int(round(SMOOTH_S * c["fps"])) | 1); x = savgol_filter(S[col].to_numpy(), w, 2)
        return spearmanr(x[:-k], x[k:]).statistic
    jit = lambda col: float(np.median(np.abs(np.diff(S[col]))[np.diff(S.frame) == 1]) / max(np.percentile(S[col], 95) - np.percentile(S[col], 5), 1e-6))
    rows.append(dict(backend=TAG, video=name, frames=len(S), variacion_fondo_pct=100 * float(np.nanmedian(cv)), autocorr_z_05s=lag("z_ave"), autocorr_znorm_05s=lag("z_norm"), jitter_rel_z=jit("z_ave"), jitter_rel_znorm=jit("z_norm"),
                     pasa_filtro=bool(100 * np.nanmedian(cv) < 5 and lag("z_ave") > 0.5)))
FIL = pd.DataFrame(rows); FIL.to_csv(OUT / "filtro.csv", index=False); display(FIL.round(3))

## 4. Trayectoria 3D

Ejes con la unidad del ancho del cuadro. z = z_norm sin reescalar. La flor es un cilindro con el radio de la mitad del ancho de su región (supuesto).

In [ ]:
AR = 1080 / 1920
def figure(name, s):
    D, inf = R[name][s], INFO[name]
    figura_3d(D, f"{name} | {s} | {TAG}", inf["ref"], inf["ref_name"], AR, OUT / f"trayectoria3d_{name}_{s}", xlim=(0.3, 0.85), ylim=(0.42, 0.08))

def strip(name, s):
    D, inf = R[name][s], INFO[name]
    tira_puntos(D, CTX[name]["cap"], inf["ref"], inf["W"], inf["H"], OUT / f"tira_{name}_{s}.jpg")

for name in VIDEO_LIST:
    for s in POINT_SETS:
        figure(name, s); strip(name, s)
    CTX[name]["cap"].release()


## 5. Comparación con S8

Las salidas de cada backend en S8 están en `resultados/S8_exploracion_3d/salidas/<backend>_<modelo>/`.

In [ ]:
REF8 = S8 / "salidas" / TAG
if REF8.exists():
    comparar(CTRL, REF8 / "control_ruido.csv", ["video", "puntos"], ["dif_entre_puntos", "ruido_vecinos", "rango_z"], nombre=f"control de ruido {TAG}")
    comparar(FIL, REF8 / "filtro.csv", ["video"], ["variacion_fondo_pct", "autocorr_z_05s"], tol=0.05, nombre=f"filtro {TAG}")
    for name in VIDEO_LIST:
        for s in POINT_SETS:
            f = REF8 / f"puntos_{name}_{s}.csv"
            if f.exists():
                comparar(R[name][s], f, ["punto"], ["z_norm", "z_norm_std"], tol=0.02, nombre=f"z_norm {name} {s}")
else:
    print("sin referencia de S8 para", TAG)

## 6. Revisión a ciegas de i5bE (D-18)

Yilber etiquetó 20 recortes de i5bE sin ver ningún mapa: *delante*, *posada delante*, *detrás* o *al lado* de la flor
(`datos_derivados/profundidad/revision_ciega/plantilla_revision_ciega.csv`). Interpretación fijada **antes** de abrir
la clave: «posada delante» = delante con posado = 1; «al lado» se excluye del acierto de signo. Acierto de signo:
delante ⇔ dz < 0. El p-valor binomial bilateral frente a 0,5 ignora que los frames son del mismo video y que se
comparan 4 modelos: es una señal, no una prueba.

Referencia: VDA Small 14/17 (p = 0,013), YOLO26s 8, YOLO26x 10, recorte 12.
**Cautela:** los «detrás» están todos en los primeros 5 s, así que el acierto puede confundirse con el tramo del video.

In [ ]:
from scipy.stats import binomtest
RC = P["dd"] / "profundidad" / "revision_ciega"
OUT_CIEGA = RUNS / "06" / "ciega"; OUT_CIEGA.mkdir(parents=True, exist_ok=True)
# Portado de video-val/resultados/profundidad/siete_puntos/comparacion_backends/ciega/evaluar_ciega.py
PLANTILLA, CLAVE = RC / "plantilla_revision_ciega.csv", RC / "clave_revision_ciega.csv"
BACKENDS = [("yolo26", "yolo26s-depth"), ("yolo26", "yolo26x-depth"), ("yolo26_crop", "yolo26s-depth"), ("vda", "vda-small")]
FLOR, FONDO = (0.54, 0.38, 0.69, 0.58), (0.30, 0.03, 0.62, 0.16)
ETQ = {"delante": ("delante", 0), "posada delante": ("delante", 1), "detras": ("detras", 0), "detrás": ("detras", 0), "al lado": ("al lado", 0)}
COLOR = {"delante": "#1f77b4", "detras": "#d62728", "al lado": "#7f7f7f"}

# Portado de video-val/resultados/profundidad/siete_puntos/comparacion_backends/ciega/evaluar_ciega.py
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; d = 1 + z ** 2 / n; c = p + z ** 2 / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    return (c - h) / d, (c + h) / d


def dz_by_backend(O):
    path = str(video_path(next(v for v in FUENTES.video_id if "i5befvb5mxg" in v)))
    t = pd.read_csv(next(TRAY.glob("trayectoria_*i5befvb5mxg*.csv"))).set_index("frame")
    cap = cv2.VideoCapture(path); n, W, H = int(cap.get(7)), int(cap.get(3)), int(cap.get(4))
    bg = np.median(np.stack([f for f in (read_frame(cap, i) for i in np.linspace(int(0.03 * n), n - 1, 60).astype(int)) if f is not None]), axis=0).astype(np.uint8)
    rect = lambda r: (int(r[0] * W), int(r[1] * H), int(r[2] * W), int(r[3] * H)); ref, far = rect(FLOR), rect(FONDO)
    res = []
    for kind, model in BACKENDS:
        tag = f"{kind}_{model}"; crop = None; fa = far
        if kind == "yolo26_crop":   # el mismo recorte que el notebook
            ob = t[~t.es_puente.astype(bool)]
            x0, x1 = min((ob.cx_px - ob.w_px / 2).min(), FLOR[0] * W) - .05 * W, max((ob.cx_px + ob.w_px / 2).max(), FLOR[2] * W) + .05 * W
            y0, y1 = min((ob.cy_px - ob.h_px / 2).min(), FONDO[1] * H) - .05 * H, max((ob.cy_px + ob.h_px / 2).max(), FLOR[3] * H) + .05 * H
            crop = (max(0, int(x0)), max(0, int(y0)), min(W, int(x1)), min(H, int(y1)))
            fa = (max(far[0], crop[0]), max(far[1], crop[1]), min(far[2], crop[2]), min(far[3], crop[3]))
        be = make_backend(kind, model, "i5bE", (H, W), crop=crop, cache_dir=CACHE_VDA)
        for r in O.itertuples():
            fr = read_frame(cap, r.frame); b = t.loc[r.frame]
            box = (max(0, int(b.cx_px - b.w_px / 2)), max(0, int(b.cy_px - b.h_px / 2)), min(W, int(b.cx_px + b.w_px / 2) + 1), min(H, int(b.cy_px + b.h_px / 2) + 1))
            m = motion_mask(fr, bg, *box); d = be.map(r.frame, fr)
            z = float(np.nanmedian(d[box[1]:box[3], box[0]:box[2]][m])) if m.sum() >= 30 else np.nan

            def out(rc):
                x0, y0, x1, y1 = rc; k = np.ones((y1 - y0, x1 - x0), bool); a0, b0, a1, b1 = max(box[0], x0), max(box[1], y0), min(box[2], x1), min(box[3], y1)
                if a1 > a0 and b1 > b0:
                    k[b0 - y0:b1 - y0, a0 - x0:a1 - x0] = False
                return float(np.nanmedian(d[y0:y1, x0:x1][k])) if k.mean() > .2 else float(np.nanmedian(d[y0:y1, x0:x1]))
            fl, fo = out(ref), out(fa)
            res.append(dict(backend=tag, id=r.id, frame=r.frame, respuesta=r.posicion, etiqueta=r.etq, posado=r.posado, z_ave=z, z_flor=fl, z_fondo=fo, dz=(z - fl) / (fo - fl), dz_sin_normalizar=z - fl))
    cap.release()
    return pd.DataFrame(res)


def metrics(g):
    d = g[g.etiqueta.isin(["delante", "detras"])].copy()
    d["ok"] = ((d.etiqueta == "delante") & (d.dz < 0)) | ((d.etiqueta == "detras") & (d.dz > 0))
    k, n = int(d.ok.sum()), len(d); lo, hi = wilson(k, n)
    dl, dt = d[d.etiqueta == "delante"].dz, d[d.etiqueta == "detras"].dz
    auc = float(np.mean([(b > a) + 0.5 * (b == a) for a in dl for b in dt]))
    no, si = d[d.posado == 0], d[d.posado == 1]
    return dict(aciertos=k, n=n, acierto=k / n, ic95_inf=lo, ic95_sup=hi, p_binomial=binomtest(k, n, 0.5).pvalue, auc=auc,
                verd_delante_pred_delante=int((dl < 0).sum()), verd_delante_pred_detras=int((dl >= 0).sum()), verd_detras_pred_delante=int((dt <= 0).sum()), verd_detras_pred_detras=int((dt > 0).sum()),
                aciertos_no_posados=int(no.ok.sum()), n_no_posados=len(no), aciertos_posados=int(si.ok.sum()), n_posados=len(si),
                dz_med_delante_no_posado=float(no[no.etiqueta == "delante"].dz.median()), dz_med_posado=float(si.dz.median()), dz_med_detras=float(dt.median()), dz_med_al_lado=float(g[g.etiqueta == "al lado"].dz.median()))


def evaluar_ciega():
    T = pd.read_csv(PLANTILLA, dtype={"posicion": str})
    if T.posicion.notna().sum() == 0:
        print("FALTA: la plantilla sigue vacía:", PLANTILLA); return
    T[["etq", "posado"]] = [ETQ[x.strip().lower()] for x in T.posicion]
    c = T.etq.value_counts().to_dict(); print("recuento:", c, "| posados:", int(T.posado.sum()))
    assert c == {"delante": 12, "detras": 5, "al lado": 3} and T.posado.sum() == 6, "el recuento no cuadra"
    R = dz_by_backend(T); R.to_csv(OUT_CIEGA / "evaluacion_ciega_por_frame.csv", index=False)
    S = pd.DataFrame([dict(backend=b, **metrics(g)) for b, g in R.groupby("backend", sort=False)]); S.to_csv(OUT_CIEGA / "evaluacion_ciega.csv", index=False)
    pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
    print(S.round(3).T.to_string())
    K = pd.read_csv(CLAVE); s = R[R.backend == "yolo26_yolo26s-depth"].merge(K[["id", "dz"]].rename(columns={"dz": "dz_clave"}), on="id")
    s["dz_sin_normalizar_vs_clave"] = s.dz_sin_normalizar - s.dz_clave; s.to_csv(OUT_CIEGA / "control_clave.csv", index=False)
    print("control clave (yolo26s): dif. absoluta mediana", round(float(s.dz_sin_normalizar_vs_clave.abs().median()), 4), "| máx.", round(float(s.dz_sin_normalizar_vs_clave.abs().max()), 4), "| mismo signo:", int((np.sign(s.dz_sin_normalizar) == np.sign(s.dz_clave)).sum()), "de", len(s))
    P = R.pivot(index=["id", "frame", "respuesta"], columns="backend", values="dz").round(3); P.to_csv(OUT_CIEGA / "dz_por_frame.csv"); print(P.reset_index().to_string(index=False))
    fig, axes = plt.subplots(1, len(BACKENDS), figsize=(5.4 * len(BACKENDS), 6))
    for a, (b, g) in zip(axes, R.groupby("backend", sort=False)):
        g = g.sort_values("dz").reset_index(drop=True); m = S[S.backend == b].iloc[0]
        a.axvline(0, color="k", lw=1)
        for i, r in g.iterrows():
            a.scatter(r.dz, i, c=COLOR[r.etiqueta], marker="s" if r.posado else "o", s=70, edgecolors="k", linewidths=0.4, zorder=3)
            a.annotate(f"#{r.id}", (r.dz, i), xytext=(6, -3), textcoords="offset points", fontsize=8)
        a.set_title(f"{b}\nsigno {m.aciertos}/{m.n} ({m.ic95_inf:.2f}-{m.ic95_sup:.2f}) | p {m.p_binomial:.3f} | AUC {m.auc:.2f}", fontsize=10); a.set_yticks([]); a.grid(axis="x", alpha=0.3); a.set_xlabel("dz (negativo = delante según el modelo)")
    for lab, col in COLOR.items():
        axes[0].scatter([], [], c=col, s=70, edgecolors="k", linewidths=0.4, label=lab)
    axes[0].scatter([], [], c="white", marker="s", s=70, edgecolors="k", label="posada"); axes[0].legend(fontsize=9, loc="lower right")
    fig.suptitle("i5bE: dz de cada frame con la etiqueta a ciegas de Yilber. Bien separado = azules a la izquierda de 0 y rojos a la derecha", fontsize=13)
    fig.tight_layout(); fig.savefig(OUT_CIEGA / "evaluacion_ciega.jpg", dpi=90); plt.close(fig)
    return S

S_CIEGA = evaluar_ciega()
comparar(S_CIEGA, S8 / "ciega" / "evaluacion_ciega.csv", ["backend"], ["aciertos", "n", "auc"], nombre="revisión a ciegas frente a S8")

## Conclusiones (S8)

- **D-13:** YOLO26-Depth cuadro a cuadro no sirve como coordenada 3D continua: solo 1 de 8 videos pasa el filtro.
- **D-14:** la profundidad se lee solo en los 7 puntos y relativa a la flor y al fondo del mismo mapa. En i5bE la
  diferencia entre puntos (~0,10) supera al ruido (~0,02).
- **D-18:** en la revisión a ciegas, VDA Small acierta 14 de 17 signos (p = 0,013) frente a 8-12 de los YOLO.
  **Decisión: otra prueba (SP-02.9)** con VDA Small en otro video de cámara fija y con los frames ciegos repartidos
  por todo el video. La 2D sigue siendo la base del logo.
- **Incremento:** pipeline de profundidad con backend intercambiable, revisión a ciegas reproducible y la variante
  `delante` de i5bE como propuesta visual.